## **EMG_Signal_Measurement_Implementation_Unit(I)**



> We will use the simplest sine wave for this experiment. Changing the time-domain waveform through overlaying signal.


> Then reconstruct the original waveform with filters. Let us step by step to experience the wonders of filters!






# 0. First import some formulas and functions we will be using!

In [ ]:
#@title
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
from scipy.fft import fft, fftfreq
from scipy import signal
import plotly.graph_objects as go
from scipy.signal import butter, lfilter

def butter_bandpass(low_cut, high_cut, fs, order=5):
    """
    Design band pass filter.

    Args:
        - low_cut  (float) : the low cutoff frequency of the filter.
        - high_cut (float) : the high cutoff frequency of the filter.
        - fs       (float) : the sampling rate.
        - order      (int) : order of the filter, by default defined to 5.
    """
    # calculate the Nyquist frequency
    nyq = 0.5 * fs

    # design filter
    low = low_cut / nyq
    high = high_cut / nyq
    b, a = butter(order, [low, high], btype='band')

    # returns the filter coefficients: numerator and denominator
    return b, a

def BandPassFilter(data, lowcut, highcut):
    fs = 1000
    order =5
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = lfilter(b, a, data)
    return y

def super_fft(data):
  # Number of sample points
  N = int(data.size)
  # sample spacing
  T = 0.001
  x = np.linspace(0.0, N*T, N, endpoint=False)
  yf = fft(data)
  xf = fftfreq(N, T)[1:N//2]
  yf_abs1 = np.abs(yf[1:N//2])
  fig = go.Figure()
  fig.add_trace(go.Scatter(x=xf, y=2.0/N * yf_abs1))
  fig.update_layout(
    title="Frequency Information of the Data",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
  fig.show()

def LowPassFilter(data, freq):
  sos = signal.butter(10, freq, 'lp', fs=1000, output='sos')
  filtered = signal.sosfilt(sos, data)
  return filtered
def HighPassFilter(data, freq):
  sos = signal.butter(10, freq, 'hp', fs=1000, output='sos')
  filtered = signal.sosfilt(sos, data)
  return filtered

# 1. Generate 3 waveforms, with frequency of 10Hz, 60Hz, 120Hz.

In [ ]:
t = np.linspace(0, 1, 1000, False)  # 1 second
signa1_10Hz = np.sin(2*np.pi*10*t)
signa1_60Hz = np.sin(2*np.pi*60*t)
signa1_120Hz = np.sin(2*np.pi*120*t)
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True)

ax1.plot(t, signa1_10Hz)
ax1.set_title('10 Hz sinusoids')
ax1.axis([0, 1, -2, 2])

ax2.plot(t, signa1_60Hz)
ax2.set_title('60 Hz sinusoids')
ax2.axis([0, 1, -2, 2])

ax3.plot(t, signa1_120Hz)
ax3.set_title('120 Hz sinusoids')
ax3.axis([0, 1, -2, 2])

ax3.set_xlabel('Time [seconds]')
plt.tight_layout()
plt.show()

In [ ]:
#Try to put signa1_10Hz, signa1_60Hz, signa1_120Hz into super_fft one by one!
super_fft(signa1_120Hz)

# 2. Overlay each waveform to produce a more complex signal for subsequent filtering!

In [ ]:
signa1_1060 = signa1_10Hz + signa1_60Hz
signa1_60120 = signa1_60Hz + signa1_120Hz
signal_mix = signa1_10Hz + signa1_60Hz + signa1_120Hz

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True)

ax1.plot(t, signa1_1060)
ax1.set_title('10 Hz sinusoids + 60 Hz sinusoids')
ax1.axis([0, 1, -2, 2])

ax2.plot(t, signa1_60120)
ax2.set_title('60 Hz sinusoids + 120 Hz sinusoids')
ax2.axis([0, 1, -2, 2])

ax3.plot(t, signal_mix)
ax3.set_title('10 Hz sinusoids + 60 Hz sinusoids + 120 Hz sinusoids')
ax3.axis([0, 1, -2, 2])

ax3.set_xlabel('Time [seconds]')
plt.tight_layout()
plt.show()

In [ ]:
#Try to put signa1_1060、signal_60120、signal_mix into super_fft one by one!
super_fft(signal_mix)

# 3. Let's use signal_mix to practice using high-pass filter!



> The frequency of the high-pass filter will filter out waveforms below that frequency!



In [ ]:
#Try changing the filter's freqency one by one to 100, 70, 20, 1 What happens when you change it to 150?
filtered1 = HighPassFilter(signal_mix, 1)
fig, (ax1) = plt.subplots(1, 1, sharex=True)

ax1.plot(t, filtered1)
ax1.set_title('Data after HighPassFilter')
ax1.axis([0, 1, -2, 2])
ax1.set_xlabel('Time [seconds]')
plt.tight_layout()
plt.show()

super_fft(filtered1)

# 4. Let's use signal_mix to practice using low-pass filter!

> The frequency of the low-pass filter will filter out waveforms above that frequency!

In [ ]:
#Try changing the filter's freqency one by one to 20, 50, 100, 150 What happens when you change it to 5?
filtered2 = LowPassFilter(signal_mix, 150)
fig, (ax1) = plt.subplots(1, 1, sharex=True)

ax1.plot(t, filtered2)
ax1.set_title('Data after LowPassFilter')
ax1.axis([0, 1, -2, 2])
ax1.set_xlabel('Time [seconds]')
plt.tight_layout()
plt.show()

super_fft(filtered2)

# 5. Let's use signal_mix to practice using band-pass filter!

> The frequencies of the band-pass filter will filter out waveforms outside of those frequencies!

In [ ]:
#Try changing the filter's freqencies one by one to (5,15), (40、80), (100、140)
filtered3 = BandPassFilter(signal_mix, 100, 140)
fig, (ax1) = plt.subplots(1, 1, sharex=True)

ax1.plot(t, filtered3)
ax1.set_title('Data after BandPassFilter')
ax1.axis([0, 1, -2, 2])
ax1.set_xlabel('Time [seconds]')
plt.tight_layout()
plt.show()

super_fft(filtered3)

# Unit Summary


---


1.   Frequency-domain can effeciently identify waveforms within time-domain waveform
2.   Filters can modify signals in most case, but there can be some distortion
3.   Filters' frequency will affect the filtered signal dramatically




